# 🧬 Tesis Doctoral: Ejecución de Experimentos en Google Colab con GPU

Este cuaderno Jupyter está optimizado para guiarte en el entrenamiento y la validación de los modelos generativos oncológicos utilizando la GPU de Google Colab (T4 o A100).

### Estructura en Drive detectada:
Tu carpeta del proyecto está en la raíz de tu Drive bajo la ruta:
`/MyDrive/datasets/`

## Paso 1: Montar Google Drive e ir al directorio de trabajo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cambiar de directorio a la carpeta del proyecto en Drive
%cd /content/drive/MyDrive/datasets/

## Paso 2: Instalar Dependencias del Entorno

In [ ]:
!pip install xgboost joblib pandas pyarrow ctgan tabpfn matplotlib seaborn scikit-learn tqdm

## Paso 3: Limpiar Checkpoints Anteriores (Seguridad)
Es vital eliminar cualquier checkpoint piloto anterior para que las dimensiones (1,001 características) no generen conflictos.

In [ ]:
!rm -rf results/checkpoints/steps_elite_borda/
!rm -f results/checkpoints/fd_shell_elite_borda.joblib

## Paso 4: Entrenar el Modelo SOTA (Forest Diffusion)
Este comando entrena los 50 pasos de difusión en paralelo para la cohorte Élite Borda.

In [ ]:
!python -u scripts/train_fd_colab.py \
    --dataset elite_borda \
    --n_t 50 \
    --device cuda \
    --mode train \
    --data_subpath datasets/results/elite_borda_training_table.parquet \
    --checkpoint_subpath datasets/results/checkpoints

## Paso 5: Entrenar el Modelo de Línea de Base (CTGAN)
Entrenamos el modelo generativo adversario clásico por 100 épocas en GPU.

In [ ]:
!python -u scripts/train_ctgan_colab.py \
    --epochs 100 \
    --batch_size 500 \
    --data_subpath datasets/results/elite_borda_training_table.parquet \
    --output_dir datasets/results

## Paso 6: Generación de 120,000 Muestras Sintéticas Definitivas (Forest Diffusion)
Utiliza la integración de Euler optimizada para generar la base sintética masiva de Forest Diffusion.

In [ ]:
!python -u scripts/generate_low_memory.py \
    --dataset elite_borda \
    --n_samples 120000 \
    --n_t 50

## Paso 7: Generación de 120,000 Muestras Sintéticas Definitivas (CTGAN)
Generamos las muestras sintéticas para el modelo clásico adversario.

In [ ]:
import joblib
import pandas as pd

print("🎲 Generando muestras con CTGAN...")
model_path = 'results/ctgan_model_elite_borda.pkl'
output_path = 'results/synthetic_samples_ctgan_elite_borda_120000.parquet'

model = joblib.load(model_path)
synthetic_df = model.sample(120000)

# Ajustar categorías discretas a enteros biológicos
for col in ['Technology_Label', 'Category']:
    if col in synthetic_df.columns:
        synthetic_df[col] = synthetic_df[col].round().clip(0, 1).astype(int)

synthetic_df.to_parquet(output_path)
print(f"✅ Guardado en {output_path}")

## Paso 8: Explicabilidad Post-Entrenamiento (Firma Jaccard)
Calcula el solapamiento Jaccard de los biomarcadores entre la data real y la sintética definitiva.

In [ ]:
!python scripts/analyze_borda_feature_importance.py \
    --real_path results/elite_borda_training_table.parquet \
    --synth_path results/synthetic_samples_elite_borda_120000.parquet

## Paso 9: Validación de la Hipótesis de Edwin
Ejecuta el experimento de control cruzado en las 5 cohortes utilizando los 120k datos sintéticos definitivos.

In [ ]:
!python scripts/evaluate_edwin_hypothesis.py \
    --real_path results/elite_borda_training_table.parquet \
    --synth_path results/synthetic_samples_elite_borda_120000.parquet

## Paso 10: Benchmark SOTA de Clasificadores (TSTR)
Entrenamos XGBoost, CatBoost, TabPFN, SVM y RF con la data generada y evaluamos contra pacientes reales.

In [ ]:
!python scripts/run_modern_benchmark_elite.py \
    --real_path results/elite_borda_training_table.parquet \
    --synth_path results/synthetic_samples_elite_borda_120000.parquet